# PRM Scoring Speed vs. Batch Size (RLHFlow)

Benchmark how `prm.score` throughput and peak GPU memory scale
with the scoring `batch_size` for RLHFlow PRM
(Llama3.1-8B-PRM-Deepseek-Data).

A **fixed** candidate set is scored at every batch size, so the
only thing varying is how many (question, answer) pairs go through
the PRM per forward pass — an apples-to-apples throughput sweep.

The candidate set is **real search-generated trajectories** from a
prior `generate_mcts_cnt` run (Llama-3.2-1B, level-4), not the
dataset's reference solutions — these are the multi-step
completions the PRM actually scores in the search loop (~7 steps,
~1200 chars each), so the timing is representative of production.

Unlike the vLLM speed benchmarks, peak GPU memory **is**
meaningful here: the PRM is an HF Transformers model whose
activation footprint grows with batch size, so the sweep shows
the speed/memory tradeoff of larger batches. (`score` only —
the v02 embed pass is benchmarked separately if needed.)

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)
logging.disable(logging.CRITICAL)

import warnings
warnings.filterwarnings("ignore")

import gc
import sys
sys.path.append("..")

import statistics

import torch

from utils.load_data import load_data_hf
from core.reward_models import RLHFlowPRM
from unittests.notebook_utils import benchmark_prm_score_batch

In [2]:
# Dataset + PRM paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = f"{base_dir}/prm800k/math_splits"

# Source of the candidate trajectories: a prior generate_mcts_cnt run.
# The raw generate_*.jsonl holds completions[question][completion] —
# full multi-step search trajectories paired with the dataset's
# problems by index.
run_name = (
    "mcts_cnt--level-4--Llama3.2-1B--tmpl-custom"
    "--bs-4--d-20--b-080--cpuct-2.0"
)
traj_path = (
    f"../results/prm800k/mcts_cnt--level-4/{run_name}"
    f"/generate_{run_name}--trial-000.jsonl"
)

# PRM to benchmark. Constructed as PRM(prm_dir); fits
# on a 32 GB V100 in fp16.
prm_name = "rlhflow"
prm_cls = RLHFlowPRM
prm_dir = f"{base_dir}/Llama3.1-8B-PRM-Deepseek-Data"

In [3]:
# Benchmark knobs
level = 4                       # MATH difficulty level (matches the run)
N_QUESTIONS = 32                 # questions to pull trajectories from
                                #   (ALL completions per question used;
                                #   ~97 pairs total for this run/trial)
batch_sizes = [1, 2, 4, 8]
num_trials = 2                  # timed score() runs per batch size
warmup = 1                      # untimed runs per batch size

In [4]:
# Build the fixed candidate set from real search trajectories.
# completions[i] are the trajectories for dataset question i; pair them
# with the problem text by index, skip questions with no completions,
# take the first N_QUESTIONS, using ALL completions for each. The SAME
# (questions, answers) grid is scored at every batch size.
import json

dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
with open(traj_path) as f:
    completions = json.load(f)["completions"]   # [question][completion]

questions, answers = [], []
for i, comps in enumerate(completions):
    cands = [c for c in comps if c]   # all non-empty completions
    if not cands:
        continue
    questions.append(dataset[i]["problem"])
    answers.append(cands)
    if len(questions) >= N_QUESTIONS:
        break

n_pairs = sum(len(a) for a in answers)
comps_per_q = [len(a) for a in answers]
n_steps = [len(c.split("\n\n")) for a in answers for c in a]
n_chars = [len(c) for a in answers for c in a]
print(f"n_questions = {len(questions)}, n_pairs = {n_pairs}")
print(f"completions/question: min={min(comps_per_q)} "
      f"max={max(comps_per_q)} mean={statistics.mean(comps_per_q):.1f}")
print(f"steps/answer: min={min(n_steps)} max={max(n_steps)} "
      f"mean={statistics.mean(n_steps):.1f}")
print(f"chars/answer: min={min(n_chars)} max={max(n_chars)} "
      f"mean={statistics.mean(n_chars):.0f}")

n_questions = 32, n_pairs = 530
completions/question: min=4 max=46 mean=16.6
steps/answer: min=1 max=20 mean=7.3
chars/answer: min=413 max=3455 mean=1162


## Run benchmark

Load RLHFlow PRM, sweep across all batch sizes, then tear down.

In [5]:
all_results = {}   # prm_name -> [(bs, n_pairs, peak_gb, times), ...]

print(f"\n########## {prm_name} ({os.path.basename(prm_dir)}) ##########")
prm = prm_cls(prm_dir)

all_results[prm_name] = benchmark_prm_score_batch(
    prm, questions, answers, batch_sizes,
    num_trials=num_trials, warmup=warmup,
)

del prm
gc.collect()
torch.cuda.empty_cache()


########## rlhflow (Llama3.1-8B-PRM-Deepseek-Data) ##########


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


=== batch_size = 1 ===
  trial 0:   81.22s total, 0.1532s/pair
  trial 1:   80.79s total, 0.1524s/pair
  peak GPU memory: 15.37 GB

=== batch_size = 2 ===
  trial 0:   78.62s total, 0.1483s/pair
  trial 1:   78.64s total, 0.1484s/pair
  peak GPU memory: 15.78 GB

=== batch_size = 4 ===
  trial 0:   82.95s total, 0.1565s/pair
  trial 1:   83.30s total, 0.1572s/pair
  peak GPU memory: 16.59 GB

=== batch_size = 8 ===
  trial 0:   89.27s total, 0.1684s/pair
  trial 1:   89.51s total, 0.1689s/pair
  peak GPU memory: 18.22 GB


## Summary

In [6]:
print(
    f"=== PRM scoring speed (level={level}, n_pairs={n_pairs}, "
    f"n_trials={num_trials}) ==="
)
header = (
    f"{'prm':<10}{'batch':>7}{'peak GB':>10}"
    f"{'mean s/trial':>14}{'std':>8}{'s/pair':>10}"
)
print(header)
print('-' * len(header))
for prm_name, rows in all_results.items():
    for bs, np_, peak_gb, times in rows:
        mean = statistics.mean(times)
        std = statistics.stdev(times) if len(times) > 1 else 0.0
        print(
            f"{prm_name:<10}{bs:>7}{peak_gb:>10.2f}"
            f"{mean:>14.2f}{std:>8.2f}{mean/np_:>10.4f}"
        )

=== PRM scoring speed (level=4, n_pairs=530, n_trials=2) ===
prm         batch   peak GB  mean s/trial     std    s/pair
-----------------------------------------------------------
rlhflow         1     15.37         81.00    0.30    0.1528
rlhflow         2     15.78         78.63    0.01    0.1484
rlhflow         4     16.59         83.12    0.25    0.1568
rlhflow         8     18.22         89.39    0.18    0.1687
